# Demanda Intermitente
### Croston, TSB y ADIDA para SKUs lentos

`Fase 3 · Video 14` — serie **Forecasting de Ventas de la A a la Z**

Los modelos anteriores asumen demanda continua. Pero el Video 7 identificó ~30% de SKUs
**intermitentes** (llenos de ceros), donde ARIMA y Prophet colapsan. Aquí su familia de métodos propia:
el tema que casi nadie enseña.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.intermittent import croston, tsb, adida
from src.metrics import mae, bias

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')
intermitentes = catalog.index[catalog['demand_profile'] == 'Intermitente']

# La intermitencia vive en la granularidad DIARIA (por SKU, sumando canales)
daily = (df.groupby(['sku_id', 'date'])['units_sold'].sum()
           .unstack('sku_id').fillna(0).sort_index())
di = daily[intermitentes]

zero_frac = (di == 0).mean()
adi = di.apply(lambda s: len(s) / max((s > 0).sum(), 1))
print(f'✅ {len(intermitentes)} SKUs intermitentes | granularidad diaria')
print(f'   Fracción de días en cero: mediana {zero_frac.median():.0%} (rango {zero_frac.min():.0%}–{zero_frac.max():.0%})')
print(f'   ADI (intervalo medio entre ventas): mediana {adi.median():.1f} días')

---
## 2. Por qué los métodos estándar fallan

Con 80–90% de días en cero, los modelos que vimos se rompen:

- **Naive (último valor):** el último día suele ser 0 → pronostica **0 para siempre** → nunca repones →
  quiebre de stock.
- **ARIMA / Prophet:** asumen una señal continua; con tantos ceros producen una tasa difusa sin base
  sólida, y son un mazo para romper una nuez.

Lo demostramos con un SKU intermitente concreto.

In [ ]:
# Elegimos un SKU intermitente representativo cuyo último día de train sea 0
H = 90
def last_train_zero(s):
    return s.iloc[:-H].iloc[-1] == 0
candidatos = [s for s in intermitentes if di[s].iloc[:-H].sum() > 0 and last_train_zero(di[s])]
SKU = max(candidatos, key=lambda s: di[s].sum()) if candidatos else intermitentes[0]
serie = di[SKU]
train, test = serie.iloc[:-H], serie.iloc[-H:]

naive_val = train.iloc[-1]
croston_rate = croston(train.values)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(serie.index, serie.values, color=BLUE, linewidth=0.8, alpha=0.7, label='demanda diaria')
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.hlines(naive_val, test.index[0], test.index[-1], color=RED, linewidth=2.5, label=f'Naive = {naive_val:.0f}')
ax.hlines(croston_rate, test.index[0], test.index[-1], color=GREEN, linewidth=2.5, label=f'Croston = {croston_rate:.2f}')
ax.set_title(f'{SKU}: el Naive pronostica {naive_val:.0f} (¡nunca repones!), Croston da una tasa útil',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Unidades/día'); ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'{SKU}: {(train==0).mean():.0%} de días en cero en train')
print(f'  Naive (último valor) = {naive_val:.0f}  → demanda real en el holdout: {test.sum():.0f} unidades')
print(f'  Croston (tasa/día)   = {croston_rate:.2f} → implica ~{croston_rate*H:.0f} unidades en {H} días')
print('  El Naive te dejaría sin stock; Croston da una señal de reposición sensata.')

---
## 3. Croston: tamaño ÷ intervalo

La idea de Croston (1972) es elegante: separa la serie en dos y suaviza cada una por separado:

- **z** = tamaño de las demandas no nulas (¿cuánto, cuando hay venta?)
- **p** = intervalo entre demandas (¿cada cuántos períodos?)

$$\text{pronóstico} = \frac{\hat z}{\hat p}$$

Ambas se actualizan con suavizado exponencial **solo en los períodos con venta**. El resultado es una tasa
estable que no se desploma a cero.

In [ ]:
fitted = croston(train.values, alpha=0.1, return_fitted=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train.values, color=BLUE, linewidth=0.8, alpha=0.6, label='demanda')
ax.plot(train.index, fitted, color=GREEN, linewidth=2, label='tasa Croston (un paso adelante)')
ax.set_title(f'{SKU}: Croston estima una tasa que sube y baja con la demanda, sin colapsar a 0',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Unidades/día'); ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print('La curva verde es la tasa esperada por día: eso alimenta el punto de reorden y el stock de')
print('seguridad. No intenta adivinar el día exacto de venta — eso es imposible y no hace falta.')

---
## 4. TSB y la obsolescencia

Croston tiene un punto ciego: **solo se actualiza cuando hay venta.** Si un SKU se descontinúa y deja de
venderse, Croston mantiene su tasa **congelada para siempre** — seguirías reponiendo un producto muerto.

**TSB** (Teunter-Syntetos-Babai) lo arregla: actualiza la **probabilidad de demanda** en *cada* período
(haya venta o no). Si dejan de llegar ventas, la probabilidad —y el pronóstico— **decaen hacia 0**. Lo
mostramos con un SKU ilustrativo que se descontinúa a mitad de camino.

In [ ]:
# Serie ilustrativa: demanda intermitente normal y luego el SKU "muere"
rng = np.random.default_rng(7)
activo = rng.binomial(1, 0.35, 120) * rng.integers(1, 6, 120)
muerto = np.zeros(80)
y_obs = np.concatenate([activo, muerto])

f_croston = croston(y_obs, alpha=0.1, return_fitted=True)
f_tsb     = tsb(y_obs, alpha=0.1, beta=0.05, return_fitted=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(y_obs)), y_obs, color='#CBD5E1', label='demanda')
ax.plot(f_croston, color=RED, linewidth=2, label='Croston (se queda congelado)')
ax.plot(f_tsb, color=GREEN, linewidth=2, label='TSB (decae hacia 0)')
ax.axvline(120, color='black', linestyle='--', linewidth=1)
ax.text(121, ax.get_ylim()[1]*0.9, 'el SKU se descontinúa →', fontsize=10)
ax.set_title('Obsolescencia: TSB reacciona, Croston no', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Tras 80 períodos sin venta:')
print(f'  Croston sigue pronosticando {f_croston[-1]:.2f}/período (repondrías un producto muerto)')
print(f'  TSB ha decaído a           {f_tsb[-1]:.2f}/período (deja de reponer)')

---
## 5. ADIDA: agregación temporal

Otra estrategia: si el problema es el exceso de ceros, **agrégalos**. ADIDA
(*Aggregate-Disaggregate Intermittent Demand Approach*) suma la demanda en bloques (p. ej. semanas),
pronostica la serie agregada —mucho menos intermitente— y luego **desagrega** el resultado a la
granularidad original. Reduce el ruido de los ceros sin un modelo especial.

In [ ]:
weekly_sku = serie.resample('W-MON').sum()
rate_adida = adida(train.values, aggregation=7, alpha=0.1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(serie.index, serie.values, color=BLUE, linewidth=0.7)
axes[0].set_title('Diario (muy intermitente)', fontsize=12, fontweight='bold')
axes[1].plot(weekly_sku.index, weekly_sku.values, color=PURPLE, linewidth=1.5)
axes[1].set_title('Agregado semanal (menos ceros → más fácil)', fontsize=12, fontweight='bold')
fig.suptitle('ADIDA: agregar reduce la intermitencia', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Ceros diarios: {(serie==0).mean():.0%}  →  ceros semanales: {(weekly_sku==0).mean():.0%}')
print(f'Tasa ADIDA (desagregada a diario): {rate_adida:.2f} unidades/día')

---
## 6. Comparación honesta de métodos

Evaluamos todos los métodos sobre los SKUs intermitentes (pronóstico de tasa plana, holdout de 90 días).
Y aquí va la **honestidad** que distingue a un especialista: en **precisión puntual (MAE)**, los métodos de
intermitencia apenas superan a una media simple — es un resultado conocido en la literatura. Su valor real
no es un MAE mágico, sino: (1) dar una **tasa** de reposición, (2) no **colapsar a 0** como el naive, y
(3) manejar la **obsolescencia** (TSB) y el **sesgo** (SBA).

In [ ]:
H = 90
methods = {
    'Naive':      lambda tr: tr.iloc[-1],
    'Media':      lambda tr: tr.mean(),
    'Croston':    lambda tr: croston(tr.values),
    'SBA':        lambda tr: croston(tr.values, variant='sba'),
    'TSB':        lambda tr: tsb(tr.values),
    'ADIDA':      lambda tr: adida(tr.values, aggregation=7),
}
rows = []
for s in intermitentes:
    ser = di[s]
    tr, te = ser.iloc[:-H], ser.iloc[-H:]
    if tr.sum() == 0:
        continue
    for name, fn in methods.items():
        f = np.full(H, fn(tr))
        rows.append({'metodo': name, 'MAE': mae(te, f), 'bias': bias(te, f)})

res = pd.DataFrame(rows).groupby('metodo').mean().reindex(list(methods))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(res.index[::-1], res['MAE'][::-1],
             color=[GREEN if v == res['MAE'].min() else BLUE for v in res['MAE'][::-1]])
axes[0].set_title('MAE medio (menor = mejor) — diferencias pequeñas', fontsize=12, fontweight='bold')
axes[1].barh(res.index[::-1], res['bias'][::-1],
             color=[RED if v > 0 else GREEN for v in res['bias'][::-1]])
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Sesgo relativo (0 = insesgado)', fontsize=12, fontweight='bold')
fig.suptitle('Métodos de intermitencia sobre los SKUs slow-movers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(res.round(3).to_string())
print('\nLectura honesta:')
print('  • En MAE, Croston/TSB/ADIDA ganan por MUY poco a la media: la precisión puntual')
print('    en intermitentes es intrínsecamente limitada (no sabes el día exacto).')
print('  • OJO al sesgo: aquí TODOS subestiman (bias < 0) porque estos SKUs CRECEN y una tasa')
print('    plana entrenada en el pasado se queda corta — ni Croston ni TSB modelan tendencia.')
print('    (En series ESTACIONARIAS, Croston sesga al alza y SBA lo corrige; aquí manda la tendencia.)')
print('  • El Naive es competitivo en MAE pero INSERVIBLE en la práctica: pronostica 0.')
print('  • El valor de estos métodos es operativo (tasa + obsolescencia), no un MAE espectacular.')

---
## 7. Conclusiones y puente al Video 15

### Las reglas que te llevas

1. **Con demanda intermitente no pronosticas el *cuándo*, sino la *tasa*.** Es lo que el inventario necesita.
2. **El Naive es una trampa:** pronostica 0 la mayoría del tiempo → quiebres de stock garantizados.
3. **Croston** estima la tasa como tamaño ÷ intervalo; **SBA** corrige su sesgo al alza (en series
   estacionarias). Ojo: **ninguno modela tendencia** — si el SKU crece, subestimarás (lo vimos en el sesgo).
4. **TSB maneja la obsolescencia:** decae a 0 cuando el SKU muere; Croston se queda congelado.
5. **ADIDA** agrega en el tiempo para reducir la intermitencia — simple y efectivo.
6. **Sé honesto con las métricas:** en MAE puntual la ventaja es marginal. El valor es operativo, no un
   número bonito. Un especialista sabe *por qué* usa el método, no solo *que* lo usa.

---

### Próximo video
**Video 15 — XGBoost y LightGBM: forecasting como problema de regresión**
El cambio de paradigma: convertir la serie en un dataset tabular (con los features de la Fase 2) y dejar
que el gradient boosting aprenda — con manejo multi-step y comparación honesta contra SARIMA.